In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import coo_matrix
from sklearn.neighbors import NearestNeighbors

In [8]:
import joblib

In [3]:
datasetA=pd.read_csv(r"D:\PRJ1\PROJECT1\data\processed\Acleaned.csv")
datasetR=pd.read_csv(r"D:\PRJ1\PROJECT1\data\processed\Rcleaned.csv")
dataset_AR=datasetR.merge(datasetA[["anime_id","name"]],on="anime_id")

In [4]:
datasetA['genre_type_combined']=(datasetA['genre']+ " " +datasetA['type'])

**FIRST ML LOGIC(content-based filtering)**

In [ ]:
tf_idf=TfidfVectorizer(stop_words="english")
tf_idf_matrix=tf_idf.fit_transform(datasetA['genre_type_combined'])
similarity_matrix=cosine_similarity(tf_idf_matrix)
top_50_similarities={}
for idx in range(similarity_matrix.shape[0]):
    similarity_scores=list(enumerate(similarity_matrix[idx]))
    similarity_scores=sorted(similarity_scores,key=lambda x: x[1],reverse=True)
    similarity_scores=similarity_scores[1:51]
    top_50_similarities[idx]=[(anime_index,float(sim_score)) for anime_index,sim_score in similarity_scores]
ind=pd.Series(datasetA.index,index=datasetA['name']).drop_duplicates()
def recomm_ani(ani_title,no_recomm=10):
    indices=ind[ani_title]
    sim_score=list(enumerate(similarity_matrix[indices]))
    sim_score=sorted(sim_score,key=lambda x: x[1],reverse=True)
    sim_score=sim_score[1:no_recomm+1]
    recomm=[]
    for ani_index,score in sim_score:
        recomm.append({"Name":datasetA.iloc[ani_index]["name"],"score":float(score)})
    return pd.DataFrame(recomm)

**SECOND ML LOGIC(collaborative filtering)**

In [6]:
ucate=dataset_AR["user_id"].astype("category")
acate=dataset_AR["name"].astype("category")
ucodes=ucate.cat.codes
acodes=acate.cat.codes
rating_values=dataset_AR["rating"].astype(np.float32)
sparse_matrix=coo_matrix((rating_values,(acodes,ucodes))).tocsr()
anime_names=acate.cat.categories
knn_model=NearestNeighbors(metric="cosine",algorithm="brute",n_neighbors=11)
knn_model.fit(sparse_matrix)
ani_to_index={anime:idx
              for idx, anime in enumerate(anime_names)}
index_to_ani={idx:anime
              for idx,anime in enumerate(anime_names)}
def knn_recc_model(ani_name,no_recc=10):
    if ani_name not in ani_to_index:
        return "Anime not found!"
    ani_indx=ani_to_index[ani_name]
    distance,index=knn_model.kneighbors(sparse_matrix[ani_indx],n_neighbors=no_recc+1)
    recco=[]
    for i in range(1,len(distance.flatten())):
        recc_index=int(index.flatten()[i])
        recc_anime=index_to_ani[recc_index]
        dist=float(distance.flatten()[i])
        collab_score=(1-dist)
        recco.append(
            {"Name":recc_anime,"Distance":collab_score}
        )
    return pd.DataFrame(recco)

**THIRD ML LOGIC(hybrid)**

In [9]:
def hybrid(ani_name,no_recomm=10,content_weight=0.25,collab_weight=0.75):
    content_dataframe=recomm_ani(ani_name,no_recomm=50)
    collab_dataframe=knn_recc_model(ani_name,no_recc=50)
    hybrid_dataframe=pd.merge(content_dataframe,collab_dataframe,on="Name",how="outer")
    hybrid_dataframe["score"]=hybrid_dataframe["score"].fillna(0)
    hybrid_dataframe["Distance"]=hybrid_dataframe["Distance"].fillna(0)
    hybrid_dataframe["Hybrid_score"]=(collab_weight*hybrid_dataframe["Distance"]+content_weight*hybrid_dataframe["score"])
    hybrid_dataframe=hybrid_dataframe.sort_values(by="Hybrid_score",ascending=False)
    return hybrid_dataframe[["Name","Hybrid_score"]].head(no_recomm)

In [7]:
knn_recc_model("Death Note").head()

,Name,Distance
0,Code Geass: Hangyaku no Lelouch,0.712688
1,Code Geass: Hangyaku no Lelouch R2,0.686494
2,Elfen Lied,0.672892
3,Shingeki no Kyojin,0.664856
4,Fullmetal Alchemist: Brotherhood,0.658528


In [8]:
recomm_ani("Death Note").head()

,Name,score
0,Mousou Dairinin,0.967700
1,Death Note Rewrite,0.943175
2,Higurashi no Naku Koro ni Kai,0.882864
3,Mirai Nikki (TV),0.822645
4,Higurashi no Naku Koro ni Rei,0.816495


In [10]:
hybrid("Death Note")

,Name,Hybrid_score
59,Mirai Nikki (TV),0.662453
3,Another,0.593962
36,Higurashi no Naku Koro ni,0.569822
16,Code Geass: Hangyaku no Lelouch,0.534516
58,Mahou Shoujo Madoka★Magica,0.524543
17,Code Geass: Hangyaku no Lelouch R2,0.514870
26,Elfen Lied,0.504669
81,Shingeki no Kyojin,0.498642
31,Fullmetal Alchemist: Brotherhood,0.493896
30,Fullmetal Alchemist,0.477936


In [10]:
joblib.dump(top_50_similarities,r"D:\PRJ1\PROJECT1\models\top_sim.pkl")

['D:\\PRJ1\\PROJECT1\\models\\top_sim.pkl']